## Naloga 4: Odprta naloga

V datoteki `dn3_cracked.csv` je podana podatkovna množica, kjer primeri ustrezajo študentom. Tvoja naloga je sestaviti napovedni model, ki glede na smiselno izbrano metriko čim bolje napove, ali je študent študij končal, ga še vedno opravlja, ali je s študijem prenehal (to so vrednosti v zadnjem stolpcu z naslovom `Target`).

Celoten postopek in vse poskuse (tudi neuspešne) zabeleži in o njih poročaj.

> Pozor: Ne predpostavljaj, da so podatki čisti. Izbira mere uspešnosti, postopek delitve podatkov in obravnava razredne neuravnoteženosti so del naloge — vse svoje odločitve dobro premisli in tudi pisno utemelji v oddani rešitvi.

Podatkovna množica vsebuje naslednje spremenljivke:

- Zakonski stan: kategorični
- Vrsta vloge: kategoričen
- Vrstni red vloge: int
- Tečaj: kategoričen
- Prisotnost podnevi/zvečer: kategorična
- Prejšnja kvalifikacija: kategorična
- Prejšnja kvalifikacija (ocena): float
- Državljanstvo: kategorialno
- Materina kvalifikacija: kategorična
- Očetova kvalifikacija: kategorična
- Poklic matere: kategoričen
- Poklic očeta: kategoričen
- Ocena pri vpisu: float
- Preseljen: kategorično
- Posebne izobraževalne potrebe: kategorično
- Dolžnik: kategorično
- Šolnine do datuma: kategorično
- Spol: kategorično
- Zadnje tri številke študentske izkaznice: int
- Štipendist: kategorično
- Starost ob vpisu: int
- Tujec: kategorično
- Učne enote 1. semester (priznane): int
- Učne enote 1. semester (vpisane): int
- Učne enote 1. semester (ocene): int
- Učne enote 1. semester (odobrene): int
- Učne enote 1. semester (ocena): float
- Učne enote 1. semester (brez ocen): int
- Učne enote 2. semester (priznane): int
- Učne enote 2. semester (vpisane): int
- Učne enote 2. semester (ocene): int
- Učne enote 2. semester (odobrene): int
- Učne enote 2. semester (ocena): float
- Učne enote 2. semester (brez ocen): int
- Stopnja brezposelnosti: float
- Stopnja inflacije: float
- BDP: float
- Standardiziran rezultat sprejemnega testa: float
- Indeks socioekonomskega statusa: float
- Anketni rezultat zadovoljstva: float
- Povprečna ocena srednje šole (normalizirana): float
- Indeks oddaljenosti od fakultete: float
- Mesečni dohodek staršev (z-score): float
- Rezultat psihološkega testa A: float
- Rezultat psihološkega testa B: float
- Indeks delovne obremenitve: float
- Število ur učenja na teden (z-score): float
- Indeks digitalne pismenosti: float
- Rezultat preverjanja angleščine: float
- Ocena motivacijskega pisma: float
- Indeks zdravstvenega stanja: float
- Rezultat testa logičnega sklepanja: float
- Številka prijave: int
- Šifra mentorja: int
- Številka razreda v srednji šoli: int
- Poštna številka: int
- Šifra študijskega programa: int
- Ciljna vrednost (`Target`): kategoričen


## Audit podatkov

Spodnje celice podatke samo berejo in povzemajo. Ne izvajajo čiščenja, imputacije, kodiranja, delitve podatkov ali modeliranja. Ker CSV nima lastnih podatkovnih tipov, audit poleg surovega zapisa prikaže še pričakovani tip iz navodil in tip, ki ga je mogoče sklepati iz nepraznih vrednosti.

In [1]:
from pathlib import Path
import csv
import math
from collections import Counter
from html import escape
from IPython.display import display, HTML

DATA_PATH = Path("dn3_cracked.csv")
MISSING_MARKERS = {"", "na", "n/a", "nan", "null", "none", "?"}

with DATA_PATH.open(encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle, delimiter=";")
    columns = list(reader.fieldnames or [])
    rows = list(reader)

def is_missing(value):
    return value is None or value.strip().lower() in MISSING_MARKERS

def as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None

def show_table(records, columns_to_show=None):
    records = list(records)
    if not records:
        display(HTML("<em>Ni vrstic za prikaz.</em>"))
        return
    columns_to_show = columns_to_show or list(records[0])
    header = "".join(f"<th>{escape(str(col))}</th>" for col in columns_to_show)
    body = []
    for record in records:
        cells = "".join(
            f"<td>{escape(str(record.get(col, '')))}</td>" for col in columns_to_show
        )
        body.append(f"<tr>{cells}</tr>")
    table = (
        "<div style='overflow-x:auto; max-height:520px'>"
        "<table><thead><tr>" + header + "</tr></thead><tbody>"
        + "".join(body) + "</tbody></table></div>"
    )
    display(HTML(table))

EXPECTED_CATEGORICAL = {
    "Marital status", "Application mode", "Course",
    "Daytime/evening attendance", "Previous qualification", "Nationality",
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation", "Displaced",
    "Educational special needs", "Debtor", "Tuition fees up to date",
    "Gender", "Scholarship holder", "International", "Target"
}
EXPECTED_INTEGER = {
    "Application order", "Last three digits of the ID", "Age at enrollment",
    "Curricular units 1st sem (credited)", "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)", "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)", "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)", "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (without evaluations)", "Application number",
    "Mentor code", "High school class number", "Postal code",
    "Study program code"
}
EXPECTED_FLOAT = set(columns) - EXPECTED_CATEGORICAL - EXPECTED_INTEGER
EXPECTED_TYPE = {
    **{col: "kategoričen" for col in EXPECTED_CATEGORICAL},
    **{col: "int" for col in EXPECTED_INTEGER},
    **{col: "float" for col in EXPECTED_FLOAT},
}

assert columns and columns[-1] == "Target", "Pričakovani zadnji stolpec Target manjka."
assert set(columns) == set(EXPECTED_TYPE), "Shema CSV-ja se ne ujema s shemo iz navodil."
print(f"Naložena datoteka: {DATA_PATH.resolve()}")
print("Podatki so shranjeni kot surovi nizi; audit jih ne spreminja.")

Naložena datoteka: C:\Users\Brus\Documents\Faks\ITAP\dn3_cracked.csv
Podatki so shranjeni kot surovi nizi; audit jih ne spreminja.


### 1. Oblika podatkov

In [2]:
print(f"Oblika (vrstice, stolpci): ({len(rows)}, {len(columns)})")
print(f"Število primerov: {len(rows)}")
print(f"Število napovednih stolpcev: {len(columns) - 1}")
print(f"Ciljni stolpec: {columns[-1]}")
print(f"Podvojena imena stolpcev: {len(columns) - len(set(columns))}")

Oblika (vrstice, stolpci): (1875, 58)
Število primerov: 1875
Število napovednih stolpcev: 57
Ciljni stolpec: Target
Podvojena imena stolpcev: 0


### 2. Tipi stolpcev

`Tip v CSV` je vedno besedilni zapis. `Sklepani logični tip` temelji samo na nepraznih vrednostih; kodirane kategorične spremenljivke ostajajo kategorične, četudi so zapisane s številkami.

In [3]:
type_rows = []
for column in columns:
    present = [row[column] for row in rows if not is_missing(row[column])]
    parsed = [as_float(value) for value in present]
    parsing_failures = sum(value is None for value in parsed)
    non_finite = sum(value is not None and not math.isfinite(value) for value in parsed)
    if column == "Target":
        inferred = "besedilo / kategorija"
    elif parsing_failures or non_finite:
        inferred = "mešano ali neštevilsko"
    elif all(value.is_integer() for value in parsed):
        inferred = "celoštevilsko kodirano" if column in EXPECTED_CATEGORICAL else "celoštevilsko"
    else:
        inferred = "število s plavajočo vejico"
    type_rows.append({
        "Stolpec": column,
        "Tip v CSV": "str",
        "Pričakovani tip": EXPECTED_TYPE[column],
        "Sklepani logični tip": inferred,
        "Neprazne vrednosti": len(present),
        "Napake pri številskem branju": parsing_failures if column != "Target" else "—",
    })
show_table(type_rows)

Stolpec,Tip v CSV,Pričakovani tip,Sklepani logični tip,Neprazne vrednosti,Napake pri številskem branju
Curricular units 2nd sem (enrolled),str,int,celoštevilsko,1654,0
Scholarship holder,str,kategoričen,celoštevilsko kodirano,1661,0
Curricular units 2nd sem (approved),str,int,celoštevilsko,1661,0
Logical reasoning test score,str,float,število s plavajočo vejico,1875,0
Debtor,str,kategoričen,celoštevilsko kodirano,1667,0
Mother's occupation,str,kategoričen,celoštevilsko kodirano,1665,0
Mother's qualification,str,kategoričen,celoštevilsko kodirano,1665,0
Application order,str,int,celoštevilsko,1666,0
Digital literacy index,str,float,število s plavajočo vejico,1875,0
Admission grade,str,float,število s plavajočo vejico,1650,0


### 3. Porazdelitev `Target`

In [4]:
target_values = ["<MANJKA>" if is_missing(row["Target"]) else row["Target"] for row in rows]
target_counts = Counter(target_values)
target_rows = [
    {
        "Razred": label,
        "Število": count,
        "Delež": f"{100 * count / len(rows):.2f}%",
    }
    for label, count in target_counts.most_common()
]
if "<MANJKA>" not in target_counts:
    target_rows.append({"Razred": "<MANJKA>", "Število": 0, "Delež": "0.00%"})
show_table(target_rows)
largest = max(target_counts.values())
smallest_nonzero = min(count for count in target_counts.values() if count > 0)
print(f"Razmerje med največjim in najmanjšim prisotnim razredom: {largest / smallest_nonzero:.2f} : 1")

Razred,Število,Delež
Dropout,1406,74.99%
Enrolled,375,20.00%
Graduate,94,5.01%
<MANJKA>,0,0.00%


Razmerje med največjim in najmanjšim prisotnim razredom: 14.96 : 1


### 4. Manjkajoče vrednosti

Kot manjkajoče se štejejo prazna polja in običajne oznake `NA`, `N/A`, `NaN`, `null`, `none` ter `?` (neobčutljivo na velikost črk).

In [5]:
missing_rows = []
for column in columns:
    count = sum(is_missing(row[column]) for row in rows)
    missing_rows.append({
        "Stolpec": column,
        "Manjka": count,
        "Delež": f"{100 * count / len(rows):.2f}%",
    })
missing_rows.sort(key=lambda record: (-record["Manjka"], record["Stolpec"]))
show_table(missing_rows)
total_missing = sum(record["Manjka"] for record in missing_rows)
total_cells = len(rows) * len(columns)
columns_with_missing = sum(record["Manjka"] > 0 for record in missing_rows)
print(f"Skupaj manjka {total_missing} od {total_cells} celic ({100 * total_missing / total_cells:.2f}%).")
print(f"Stolpci z vsaj eno manjkajočo vrednostjo: {columns_with_missing} od {len(columns)}.")

Stolpec,Manjka,Delež
Admission grade,225,12.00%
Daytime/evening attendance,223,11.89%
Curricular units 2nd sem (credited),222,11.84%
Curricular units 2nd sem (enrolled),221,11.79%
Curricular units 1st sem (grade),220,11.73%
Curricular units 2nd sem (evaluations),219,11.68%
Course,218,11.63%
Gender,217,11.57%
Application mode,216,11.52%
Father's qualification,215,11.47%


Skupaj manjka 8415 od 108750 celic (7.74%).
Stolpci z vsaj eno manjkajočo vrednostjo: 40 od 58.


### 5. Duplikati

Preverjanje je izvedeno na celotnih surovih vrsticah, zato so duplikati samo popolnoma enaki zapisi v vseh 58 stolpcih.

In [6]:
signatures = [tuple(row[column] for column in columns) for row in rows]
signature_counts = Counter(signatures)
duplicate_excess = sum(count - 1 for count in signature_counts.values() if count > 1)
duplicate_groups = sum(count > 1 for count in signature_counts.values())
print(f"Popolnoma podvojene vrstice (brez prve pojavitve): {duplicate_excess}")
print(f"Skupine podvojenih vrstic: {duplicate_groups}")
print(f"Podvojena imena stolpcev: {len(columns) - len(set(columns))}")

Popolnoma podvojene vrstice (brez prve pojavitve): 0
Skupine podvojenih vrstic: 0
Podvojena imena stolpcev: 0


### 6. Neveljavne in notranje nedosledne vrednosti

Spodnja pravila so namenoma konservativna: preverjajo le zanesljivo določljive napake iz navodil in osnovne notranje relacije. Manjkajoče vrednosti se poročajo ločeno in tu niso označene kot neveljavne. Brez uradnega šifranta ni mogoče zanesljivo presoditi, ali je posamezna številčna koda kategorije dovoljena.

In [7]:
validation_results = []

def add_validation(rule, failures):
    failures = list(failures)
    validation_results.append({
        "Pravilo": rule,
        "Kršitve": len(failures),
        "Primeri": "; ".join(failures[:5]) if failures else "—",
    })

numeric_columns = set(columns) - {"Target"}
failures = []
for csv_row, row in enumerate(rows, start=2):
    for column in numeric_columns:
        if is_missing(row[column]):
            continue
        value = as_float(row[column])
        if value is None:
            failures.append(f"vrstica {csv_row}, {column}={row[column]!r}")
add_validation("Neštevilski zapis v številskem ali številčno kodiranem stolpcu", failures)

failures = []
for csv_row, row in enumerate(rows, start=2):
    for column in numeric_columns:
        if is_missing(row[column]):
            continue
        value = as_float(row[column])
        if value is not None and not math.isfinite(value):
            failures.append(f"vrstica {csv_row}, {column}={row[column]!r}")
add_validation("Neskončna ali NaN številska vrednost", failures)

integer_like_columns = EXPECTED_INTEGER | (EXPECTED_CATEGORICAL - {"Target"})
failures = []
for csv_row, row in enumerate(rows, start=2):
    for column in integer_like_columns:
        if is_missing(row[column]):
            continue
        value = as_float(row[column])
        if value is not None and math.isfinite(value) and not value.is_integer():
            failures.append(f"vrstica {csv_row}, {column}={row[column]!r}")
add_validation("Necela vrednost v celoštevilskem ali kategorično kodiranem stolpcu", failures)

allowed_targets = {"Dropout", "Enrolled", "Graduate"}
failures = [
    f"vrstica {csv_row}, Target={row['Target']!r}"
    for csv_row, row in enumerate(rows, start=2)
    if not is_missing(row["Target"]) and row["Target"] not in allowed_targets
]
add_validation("Target ni eden od Dropout, Enrolled, Graduate", failures)

binary_columns = {
    "Daytime/evening attendance", "Displaced", "Educational special needs",
    "Debtor", "Tuition fees up to date", "Gender",
    "Scholarship holder", "International"
}
failures = []
for csv_row, row in enumerate(rows, start=2):
    for column in binary_columns:
        if is_missing(row[column]):
            continue
        value = as_float(row[column])
        if value is not None and value not in {0.0, 1.0}:
            failures.append(f"vrstica {csv_row}, {column}={row[column]!r}")
add_validation("Nebinarna vrednost v binarnem stolpcu", failures)

grade_columns = {
    "Previous qualification (grade)", "Admission grade",
    "Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)"
}
nonnegative_columns = EXPECTED_INTEGER | (EXPECTED_CATEGORICAL - {"Target"}) | grade_columns | {"Unemployment rate"}
failures = []
for csv_row, row in enumerate(rows, start=2):
    for column in nonnegative_columns:
        if is_missing(row[column]):
            continue
        value = as_float(row[column])
        if value is not None and math.isfinite(value) and value < 0:
            failures.append(f"vrstica {csv_row}, {column}={row[column]!r}")
add_validation("Negativna vrednost v stolpcu, ki mora biti nenegativen", failures)

bounds = {
    "Age at enrollment": (15, 100),
    "Last three digits of the ID": (0, 999),
    "Admission grade": (0, 200),
    "Previous qualification (grade)": (0, 200),
    "Curricular units 1st sem (grade)": (0, 20),
    "Curricular units 2nd sem (grade)": (0, 20),
    "Unemployment rate": (0, 100),
}
failures = []
for csv_row, row in enumerate(rows, start=2):
    for column, (lower, upper) in bounds.items():
        if is_missing(row[column]):
            continue
        value = as_float(row[column])
        if value is not None and math.isfinite(value) and not lower <= value <= upper:
            failures.append(f"vrstica {csv_row}, {column}={row[column]!r}, meja=[{lower}, {upper}]")
add_validation("Vrednost zunaj konservativnih stvarnih meja", failures)

failures = []
for csv_row, row in enumerate(rows, start=2):
    for semester in ("1st", "2nd"):
        enrolled_col = f"Curricular units {semester} sem (enrolled)"
        for kind in ("approved", "credited"):
            compared_col = f"Curricular units {semester} sem ({kind})"
            if is_missing(row[enrolled_col]) or is_missing(row[compared_col]):
                continue
            enrolled = as_float(row[enrolled_col])
            compared = as_float(row[compared_col])
            if enrolled is not None and compared is not None and compared > enrolled:
                failures.append(
                    f"vrstica {csv_row}, {compared_col}={compared:g} > {enrolled_col}={enrolled:g}"
                )
add_validation("Odobrene ali priznane enote presegajo vpisane enote", failures)

show_table(validation_results)
print(f"Skupno število kršitev navedenih pravil: {sum(item['Kršitve'] for item in validation_results)}")

Pravilo,Kršitve,Primeri
Neštevilski zapis v številskem ali številčno kodiranem stolpcu,0,—
Neskončna ali NaN številska vrednost,0,—
Necela vrednost v celoštevilskem ali kategorično kodiranem stolpcu,0,—
"Target ni eden od Dropout, Enrolled, Graduate",0,—
Nebinarna vrednost v binarnem stolpcu,0,—
"Negativna vrednost v stolpcu, ki mora biti nenegativen",0,—
Vrednost zunaj konservativnih stvarnih meja,0,—
Odobrene ali priznane enote presegajo vpisane enote,0,—


Skupno število kršitev navedenih pravil: 0


### 7. Kardinalnost

Kardinalnost je število različnih nepraznih vrednosti. Visoka kardinalnost je pri zveznih meritvah pričakovana; sama po sebi zato še ne pomeni, da je stolpec identifikator.

In [8]:
cardinality_rows = []
for column in columns:
    present = [row[column] for row in rows if not is_missing(row[column])]
    counts = Counter(present)
    unique_count = len(counts)
    dominant_count = max(counts.values()) if counts else 0
    cardinality_rows.append({
        "Stolpec": column,
        "Različne neprazne": unique_count,
        "Neprazne": len(present),
        "Razmerje enoličnosti": f"{unique_count / len(present):.4f}" if present else "—",
        "Največja frekvenca ene vrednosti": dominant_count,
    })
cardinality_rows.sort(
    key=lambda record: (-record["Različne neprazne"], record["Stolpec"])
)
show_table(cardinality_rows)

Stolpec,Različne neprazne,Neprazne,Razmerje enoličnosti,Največja frekvenca ene vrednosti
Digital literacy index,1875,1875,1.0000,1
Distance to faculty index,1875,1875,1.0000,1
English proficiency test score,1875,1875,1.0000,1
Health status index,1875,1875,1.0000,1
High school GPA (normalized),1875,1875,1.0000,1
Logical reasoning test score,1875,1875,1.0000,1
Motivation letter score,1875,1875,1.0000,1
Parents' monthly income (z-score),1875,1875,1.0000,1
Psychological test A score,1875,1875,1.0000,1
Satisfaction survey score,1875,1875,1.0000,1


### 8. Možni identifikatorji

Kandidati so izbrani po pomenu imena (`ID`, `number`, `code`, `postal`, `digits`) in nato opisani z enoličnostjo. Gre za opozorilo za kasnejšo presojo, ne za samodejno odločitev o odstranitvi stolpca.

In [9]:
identifier_cues = (" id", "number", "code", "postal", "digits")
identifier_rows = []
for column in columns:
    normalized_name = " " + column.lower()
    if not any(cue in normalized_name for cue in identifier_cues):
        continue
    present = [row[column] for row in rows if not is_missing(row[column])]
    counts = Counter(present)
    unique_count = len(counts)
    ratio = unique_count / len(present) if present else 0
    duplicate_excess = sum(count - 1 for count in counts.values() if count > 1)
    reason = "ime nakazuje identifikator"
    if ratio >= 0.95:
        reason += "; skoraj vsaka neprazna vrednost je enolična"
    else:
        reason += "; lahko gre za delni ali skupinski identifikator"
    identifier_rows.append({
        "Stolpec": column,
        "Manjka": len(rows) - len(present),
        "Različne neprazne": unique_count,
        "Razmerje enoličnosti": f"{ratio:.4f}",
        "Ponovitve nad prvo pojavitvijo": duplicate_excess,
        "Razlog za opozorilo": reason,
    })
identifier_rows.sort(
    key=lambda record: (-float(record["Razmerje enoličnosti"]), record["Stolpec"])
)
show_table(identifier_rows)

Stolpec,Manjka,Različne neprazne,Razmerje enoličnosti,Ponovitve nad prvo pojavitvijo,Razlog za opozorilo
High school class number,0,1860,0.9920,15,ime nakazuje identifikator; skoraj vsaka neprazna vrednost je enolična
Postal code,0,1859,0.9915,16,ime nakazuje identifikator; skoraj vsaka neprazna vrednost je enolična
Mentor code,187,1673,0.9911,15,ime nakazuje identifikator; skoraj vsaka neprazna vrednost je enolična
Application number,0,1856,0.9899,19,ime nakazuje identifikator; skoraj vsaka neprazna vrednost je enolična
Study program code,0,1853,0.9883,22,ime nakazuje identifikator; skoraj vsaka neprazna vrednost je enolična
Last three digits of the ID,187,100,0.0592,1588,ime nakazuje identifikator; lahko gre za delni ali skupinski identifikator


## Ugotovitve audita

- Podatkovna množica ima **1.875 vrstic in 58 stolpcev**: 57 napovednih spremenljivk ter ciljni stolpec `Target`. Imena stolpcev se ne podvajajo.
- Vsi napovedni stolpci so v CSV-ju zapisani številčno oziroma s številčnimi kodami, `Target` pa besedilno. Neprazne vrednosti so skladne z osnovnimi tipi iz navodil: audit ni našel neštevilskih zapisov, neskončnosti ali neceloštevilskih vrednosti v celoštevilskih oziroma kategorično kodiranih stolpcih.
- `Target` nima manjkajočih ali neznanih oznak, vendar je **močno neuravnotežen**: `Dropout` 1.406 (74,99 %), `Enrolled` 375 (20,00 %) in `Graduate` 94 (5,01 %). Največji razred je približno 14,96-krat večji od najmanjšega.
- Manjka **8.415 od 108.750 celic (7,74 %)**. Vsaj ena vrednost manjka v 40 od 58 stolpcev. Največ jih manjka v `Admission grade` (225 oziroma 12,00 %), sledijo `Daytime/evening attendance` (223 oziroma 11,89 %), `Curricular units 2nd sem (credited)` (222 oziroma 11,84 %) in `Curricular units 2nd sem (enrolled)` (221 oziroma 11,79 %).
- Popolnoma podvojenih vrstic ni.
- Konservativna preverjanja niso našla neveljavnih vrednosti: dovoljeni razredi `Target`, binarne kode, celoštevilskost, končnost, nenegativnost izbranih spremenljivk, stvarne meje za starost in ocene ter relacije med vpisanimi, priznanimi in odobrenimi enotami so brez kršitev. To ne potrjuje veljavnosti posameznih kategoričnih kod, ker navodila ne vsebujejo njihovega uradnega šifranta.
- Kot možni identifikatorji ali kvazi-identifikatorji izstopajo `Application number` (razmerje enoličnosti 0,9899), `Mentor code` (0,9911 med nepraznimi vrednostmi), `High school class number` (0,9920), `Postal code` (0,9915), `Study program code` (0,9883) in po imenu tudi `Last three digits of the ID` (0,0592). Pred modeliranjem bo treba preveriti njihov pomen in možnost uhajanja informacije; tukaj niso odstranjeni ali spremenjeni.
- Več zveznih meritev ima kardinalnost 1.875, kar je pri zveznih podatkih pričakovano in samo po sebi ni dokaz identifikatorske vloge.

V tej fazi ni bilo izvedeno nobeno čiščenje, nadomeščanje manjkajočih vrednosti, kodiranje, izločanje stolpcev, delitev podatkov ali modeliranje.